In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from tqdm import tqdm

In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 5.3 MB/s eta 0:00:00


In [ ]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train = df.iloc[:train_end].reset_index(drop=True)
df_val = df.iloc[train_end:val_end].reset_index(drop=True)
df_test = df.iloc[val_end:].reset_index(drop=True)

In [ ]:
#Разбиение на 5 фолдов (примерно по 6 месяцев)
tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

# Количество точек в горизонте прогнозов
horizons = {
    "4h":  8,
    "8h":  16,
    "24h": 48,
    "7d":  336,
    "14d": 672,
    "1m":  1488,
}

In [ ]:
# Naive модель: прогноз = последнее известное значение
# для горизонта из N точек просто повторяем последнее значение N раз

naive_results = {}

for house in houses:
    house_results = {}

    for horizon_name, horizon_points in horizons.items():
        fold_metrics = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
            # последнее значение train
            last_value = df[house].iloc[train_idx[-1]]

            # прогноз - просто повторяем последнее значение
            y_pred = np.full(horizon_points, last_value)

            # берём первые horizon_points точек из val
            y_true = df[house].iloc[val_idx[:horizon_points]].values

            # если val короче горизонта - пропускаем
            if len(y_true) < horizon_points:
                continue

            metrics = evaluate(y_true, y_pred)
            fold_metrics.append(metrics)

        # среднее по фолдам
        house_results[horizon_name] = {
            "MAE":  round(np.mean([m["MAE"]  for m in fold_metrics]), 3),
            "RMSE": round(np.mean([m["RMSE"] for m in fold_metrics]), 3),
            "MAPE": round(np.mean([m["MAPE"] for m in fold_metrics]), 3),
        }

    naive_results[house] = house_results

In [ ]:
rows = []
for house in houses:
    for horizon_name, metrics in naive_results[house].items():
        rows.append({
            "Модель": "Naive",
            "Дом": house,
            "Горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_naive = pd.DataFrame(rows)

In [ ]:
for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_naive.pivot(index="Горизонт", columns="Дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    print(f"Naive - {metric}:")
    print(pivot.to_string())
    print()

Naive - MAE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
Горизонт                                                                        
4h         11.824    4.649    2.983    7.084    3.524   13.008   15.405    5.055
8h         17.318    7.552    4.763   11.056    5.933   21.029   27.452    8.813
24h        15.663    7.190    4.615    9.950    5.931   20.849   24.907    9.510
7d         17.383    7.839    4.852   10.525    6.572   23.048   27.833   10.070
14d        17.331    7.770    4.798   10.361    6.542   23.132   27.663   10.009
1m         17.380    7.919    4.775   10.472    6.573   23.056   27.922   10.285

Naive - RMSE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
Горизонт                                                                        
4h         13.395    5.375    3.312    8.439    4.256   14.692   17.680    6.036
8h         19.715    8.671    5.483   12.730    6.943   24.649   32.474   10.401


In [ ]:
df_naive.to_csv("results_naive.csv", index=False)

In [ ]:
train_idx, val_idx = list(tscv.split(df))[-1]
house = "house_1"

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    last_value = df[house].iloc[train_idx[-1]]
    y_pred = np.full(horizon_points, last_value)
    y_true = df[house].iloc[val_idx[:horizon_points]].values
    timestamps = df["timestamp"].iloc[val_idx[:horizon_points]]

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_true,
        mode="lines", name="Фактические значения",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_pred,
        mode="lines", name="прогноз Naive",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title="Naive: Прогнозные и фактические значения по всем горизонтам прогнозирования (дом 1, fold 5)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("27b_naive_forecast_all_horizons.png", scale=2)

Наглядно видно слабость Naive — прямая горизонтальная линия совершенно не улавливает никакой график

In [ ]:
# Seasonal Naive: прогноз значения электрической нагрузки того же времени неделю назад
# лаг 336 = 7 дней * 48 интервалов

seasonal_naive_results = {}
lag = 336

for house in houses:
    house_results = {}

    for horizon_name, horizon_points in horizons.items():
        fold_metrics = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
            # берём первые horizon_points точек из val
            val_points = min(horizon_points, len(val_idx))
            y_true = df[house].iloc[val_idx[:val_points]].values

            if len(y_true) < horizon_points:
                continue

            # прогноз = значения неделю назад от начала val
            start_idx = val_idx[0]
            y_pred = df[house].iloc[start_idx - lag: start_idx - lag + horizon_points].values

            if len(y_pred) < horizon_points:
                continue

            metrics = evaluate(y_true, y_pred)
            fold_metrics.append(metrics)

        house_results[horizon_name] = {
            "MAE":  round(np.mean([m["MAE"]  for m in fold_metrics]), 3),
            "RMSE": round(np.mean([m["RMSE"] for m in fold_metrics]), 3),
            "MAPE": round(np.mean([m["MAPE"] for m in fold_metrics]), 3),
        }

    seasonal_naive_results[house] = house_results

In [ ]:
rows = []
for house in houses:
    for horizon_name, metrics in seasonal_naive_results[house].items():
        rows.append({
            "Модель": "Seasonal Naive",
            "Дом": house,
            "Горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_seasonal_naive = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_seasonal_naive.pivot(index="Горизонт", columns="Дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    print(f"Seasonal Naive - {metric}:")
    print(pivot.to_string())
    print()

df_seasonal_naive.to_csv("results_seasonal_naive.csv", index=False)

Seasonal Naive - MAE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
Горизонт                                                                        
4h          3.755    2.747    1.384    2.501    2.919    5.573    4.459    2.870
8h          3.576    2.318    1.323    2.654    2.361    5.636    4.417    2.215
24h         3.179    2.039    1.363    2.554    2.080    5.144    5.847    2.215
7d          3.563    1.921    1.537    2.465    2.046    5.194    5.575    2.331
14d         3.497    1.884    1.513    2.443    2.065    5.253    4.835    2.270
1m          3.372    1.860    1.499    2.435    2.001    5.474    4.840    2.363

Seasonal Naive - RMSE:
Дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
Горизонт                                                                        
4h          4.463    3.207    1.606    2.973    3.385    6.105    5.450    3.350
8h          4.228    2.757    1.599    3.149    2.876    6.439 

Seasonal Naive значительно лучше простого Naive — MAPE 4-11% против 14-42%.

In [ ]:
train_idx, val_idx = list(tscv.split(df))[-1]
house = "house_1"
lag = 336

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    start_idx = val_idx[0]
    y_pred = df[house].iloc[start_idx - lag: start_idx - lag + horizon_points].values
    y_true = df[house].iloc[val_idx[:horizon_points]].values
    timestamps = df["timestamp"].iloc[val_idx[:horizon_points]]

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_true,
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_pred,
        mode="lines", name="прогноз Seasonal Naive",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title="Seasonal Naive: Прогнозные и фактические значения по всем горизонтам прогнозирования (дом 1, fold 5)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("28b_seasonal_naive_forecast_all_horizons.png", scale=2)

Разница с Naive очевидна - Seasonal Naive хорошо улавливает суточный профиль. Прогноз (красный) довольно точно следует за фактом (синий), особенно в ночные часы и на спаде.

In [ ]:
!pip install workalendar -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 23.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 1.9 MB/s eta 0:00:00


In [ ]:
from workalendar.europe import Russia

cal = Russia()

def is_holiday(dt):
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)
print(f"Праздничных и выходных дней: {df['is_holiday'].sum()}")

Праздничных и выходных дней: 11519


In [ ]:
def is_holiday(dt):

    weekday = dt.weekday()
    if weekday >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

In [ ]:
from sklearn.linear_model import LinearRegression

def make_features(df, house, lag_features=[1, 2, 48, 96, 336], weather=True):
    data = df[["timestamp", house]].copy()

    data["hour"] = data["timestamp"].dt.hour
    data["weekday"] = data["timestamp"].dt.weekday
    data["month"] = data["timestamp"].dt.month
    data["is_weekend"] = (data["weekday"] >= 5).astype(int)
    data["is_holiday"] = df["is_holiday"].values

    for lag in lag_features:
        data[f"lag_{lag}"] = data[house].shift(lag)

    if weather:
        data["temp_c"] = df["temp_c"]
        data["humidity"] = df["humidity"]
        data["cloudiness"] = df["cloudiness"]

    data = data.dropna().reset_index(drop=True)

    feature_cols = [c for c in data.columns if c not in ["timestamp", house]]
    X = data[feature_cols]
    y = data[house]

    return X, y, data["timestamp"]


In [ ]:
X, y, ts = make_features(df, "house_1")
print(f"Признаков: {X.shape[1]}")
print(f"Строк: {X.shape[0]}")
print(f"Список признаков: {X.columns.tolist()}")

Признаков: 13
Строк: 35924
Список признаков: ['hour', 'weekday', 'month', 'is_weekend', 'is_holiday', 'lag_1', 'lag_2', 'lag_48', 'lag_96', 'lag_336', 'temp_c', 'humidity', 'cloudiness']


In [ ]:
lr_results = {}

for house in tqdm(houses):
    house_results = {}
    X_full, y_full, ts_full = make_features(df, house)

    for horizon_name, horizon_points in horizons.items():
        fold_metrics = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
            X_train = X_full.iloc[train_idx]
            y_train = y_full.iloc[train_idx]
            X_val = X_full.iloc[val_idx[:horizon_points]]
            y_val = y_full.iloc[val_idx[:horizon_points]]

            if len(y_val) < horizon_points:
                continue

            model = LinearRegression()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            metrics = evaluate(y_val.values, y_pred)
            fold_metrics.append(metrics)

        house_results[horizon_name] = {
            "MAE": round(np.mean([m["MAE"]  for m in fold_metrics]), 3),
            "RMSE": round(np.mean([m["RMSE"] for m in fold_metrics]), 3),
            "MAPE": round(np.mean([m["MAPE"] for m in fold_metrics]), 3),
        }

    lr_results[house] = house_results

100%|██████████| 8/8 [00:06<00:00,  1.21it/s]


In [ ]:
rows = []
for house in houses:
    for horizon_name, metrics in lr_results[house].items():
        rows.append({
            "модель": "Linear Regression",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_lr = pd.DataFrame(rows)

In [ ]:
for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_lr.pivot(index="горизонт", columns="дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    print(f"Linear Regression - {metric}:")
    print(pivot.to_string())
    print()

df_lr.to_csv("results_lr.csv", index=False)

Linear Regression - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          2.067    1.115    1.116    1.544    1.237    3.267    3.221    1.581
8h          2.178    1.175    0.983    1.666    1.319    3.140    3.496    1.653
24h         2.032    1.159    0.953    1.615    1.244    2.929    2.994    1.534
7d          2.215    1.255    0.953    1.609    1.294    2.837    2.943    1.414
14d         2.213    1.228    0.962    1.610    1.294    2.922    2.988    1.417
1m          2.181    1.229    0.957    1.601    1.278    2.907    2.991    1.399

Linear Regression - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          2.544    1.459    1.385    1.842    1.512    3.957    3.922    1.889
8h          2.674    1.527    1.239    2.044    1.651    

Linear Regression заметно лучше Seasonal Naive:

MAPE снизился с 4-10% до 3-7%
MAE заметно меньше

In [ ]:
train_idx, val_idx = list(tscv.split(X_full))[-1]

house = "house_1"
X_full, y_full, ts_full = make_features(df, house)

X_train = X_full.iloc[train_idx]
y_train = y_full.iloc[train_idx]

model = LinearRegression()
model.fit(X_train, y_train)

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    X_val = X_full.iloc[val_idx[:horizon_points]]
    y_val = y_full.iloc[val_idx[:horizon_points]]
    timestamps = ts_full.iloc[val_idx[:horizon_points]]
    y_pred = model.predict(X_val)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_val.values,
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=y_pred,
        mode="lines", name="Прогноз LR",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title="Linear Regression: Прогнозные и фактические значения по всем горизонтам прогнозирования (дом 1, fold 5)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("29_lr_forecast_all_horizons.png", scale=2)

Linear Regression работает хорошо на всех горизонтах - красная и синяя линии почти совпадают даже на 14д и 1м. Это хороший результат для линейной модели.